# ai question case

1. Read Pinecone index
2. For a query, find relevant documents
3. Get the answer

In [1]:
import openai
import os
from langchain.vectorstores import Pinecone
from langchain.chains.question_answering import load_qa_chain
from dotenv import load_dotenv, find_dotenv
import pinecone

_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key  = os.getenv('OPENAI_API_KEY')

/home/mark/anaconda3/envs/CasifyAI/lib/python3.9/site-packages/pinecone/index.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [2]:
namespace = "458"
model = "gpt-4"
index_name= "casify"

# initialize pinecone
pinecone.init(

    api_key=os.getenv('PINECONE_API_KEY'),     
    environment="us-west4-gcp"  # next to api key in console
)

In [3]:
import openai
from langchain.embeddings.openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(openai_api_key=openai.api_key)

In [4]:
def get_similar_docs(query,namespace,num_sources=5,score=False):
  index = Pinecone.from_existing_index(index_name, embeddings, namespace = namespace)
  if score:
    similar_docs = index.similarity_search_with_score(query, k=num_sources, namespace=namespace)
  else:
    similar_docs = index.similarity_search(query,k=num_sources, namespace=namespace)
  return similar_docs

In [5]:
def get_answer(query, namespace, history=None):
  from langchain.llms import OpenAI
  llm = OpenAI(model_name=model)
  chain = load_qa_chain(llm, chain_type="stuff")
  similar_docs_list = get_similar_docs(query, namespace=namespace)
  if history is None:
    print("No history provided")
    my_answer = chain.run(input_documents=similar_docs_list, question=query)
  else:
    print("Yes history provided")
    my_answer = chain.run(input_documents=similar_docs_list, question=query)    
  return my_answer

In [8]:
answer=get_answer("What are the charges?", namespace, history=None)
print(answer)

UnauthorizedException: (401)
Reason: Unauthorized
HTTP response headers: HTTPHeaderDict({'www-authenticate': 'API key is missing or invalid for the environment "us-west4-gcp". Check that the correct environment is specified.', 'content-length': '114', 'date': 'Wed, 25 Oct 2023 06:11:46 GMT', 'server': 'envoy'})
HTTP response body: API key is missing or invalid for the environment "us-west4-gcp". Check that the correct environment is specified.
